# Lab | Pandas

In this lab, we will be working with the customer data from an insurance company, which can be found in the CSV file located at the following link: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

The data includes information such as customer ID, state, gender, education, income, and other variables that can be used to perform various analyses.

Throughout the lab, we will be using the pandas library in Python to manipulate and analyze the data. Pandas is a powerful library that provides various data manipulation and analysis tools, including the ability to load and manipulate data from a variety of sources, including CSV files.

### Data Description

- Customer - Customer ID

- ST - State where customers live

- Gender - Gender of the customer

- Education - Background education of customers 

- Customer Lifetime Value - Customer lifetime value(CLV) is the total revenue the client will derive from their entire relationship with a customer. In other words, is the predicted or calculated value of a customer over their entire duration as a policyholder with the insurance company. It is an estimation of the net profit that the insurance company expects to generate from a customer throughout their relationship with the company. Customer Lifetime Value takes into account factors such as the duration of the customer's policy, premium payments, claim history, renewal likelihood, and potential additional services or products the customer may purchase. It helps insurers assess the long-term profitability and value associated with retaining a particular customer.

- Income - Customers income

- Monthly Premium Auto - Amount of money the customer pays on a monthly basis as a premium for their auto insurance coverage. It represents the recurring cost that the insured person must pay to maintain their insurance policy and receive coverage for potential damages, accidents, or other covered events related to their vehicle.

- Number of Open Complaints - Number of complaints the customer opened

- Policy Type - There are three type of policies in car insurance (Corporate Auto, Personal Auto, and Special Auto)

- Vehicle Class - Type of vehicle classes that customers have Two-Door Car, Four-Door Car SUV, Luxury SUV, Sports Car, and Luxury Car

- Total Claim Amount - the sum of all claims made by the customer. It represents the total monetary value of all approved claims for incidents such as accidents, theft, vandalism, or other covered events.


External Resources: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

## Challenge 1: Understanding the data

In this challenge, you will use pandas to explore a given dataset. Your task is to gain a deep understanding of the data by analyzing its characteristics, dimensions, and statistical properties.

- Identify the dimensions of the dataset by determining the number of rows and columns it contains.
- Determine the data types of each column and evaluate whether they are appropriate for the nature of the variable. You should also provide suggestions for fixing any incorrect data types.
- Identify the number of unique values for each column and determine which columns appear to be categorical. You should also describe the unique values of each categorical column and the range of values for numerical columns, and give your insights.
- Compute summary statistics such as mean, median, mode, standard deviation, and quartiles to understand the central tendency and distribution of the data for numerical columns. You should also provide your conclusions based on these summary statistics.
- Compute summary statistics for categorical columns and providing your conclusions based on these statistics.

In [71]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
df = pd.read_csv(url)

df.columns = [col.lower().replace(" ", "_") for col in df.columns]

# Mostramos el resultado de los nuevos nombres y las primeras filas
print("Nuevas columnas:", df.columns.tolist())
df.head()

Nuevas columnas: ['customer', 'st', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'vehicle_class', 'total_claim_amount']


,customer,st,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


In [72]:
# 1. Dimensiones
print(f"La tabla tiene {df.shape[0]} filas y {df.shape[1]} columnas.\n")

# 2. Tipos de datos y conteo de nulos
print("Resumen de datos y valores faltantes:")
df.info()

La tabla tiene 4008 filas y 11 columnas.

Resumen de datos y valores faltantes:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4008 entries, 0 to 4007
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer                   1071 non-null   object 
 1   st                         1071 non-null   object 
 2   gender                     954 non-null    object 
 3   education                  1071 non-null   object 
 4   customer_lifetime_value    1068 non-null   object 
 5   income                     1071 non-null   float64
 6   monthly_premium_auto       1071 non-null   float64
 7   number_of_open_complaints  1071 non-null   object 
 8   policy_type                1071 non-null   object 
 9   vehicle_class              1071 non-null   object 
 10  total_claim_amount         1071 non-null   float64
dtypes: float64(3), object(8)
memory usage: 344.6+ KB


In [73]:
# en info, encontramos 3 problemas:
# - filas vacias - simbolos raros - datos raros en columnas numéricas
# 1. Borramos filas donde TODO sea nulo
df = df.dropna(how='all')

# 2. Borramos duplicados
df = df.drop_duplicates()

print(f"Ahora nos quedan {df.shape[0]} filas.")

Ahora nos quedan 1071 filas.


In [74]:
# 1. Quitamos el '%' y convertimos a número
# Usamos .str para tratar la columna como texto momentáneamente
df['customer_lifetime_value'] = df['customer_lifetime_value'].str.replace('%', '')

# 2. Convertimos a numérico (los errores los pone como NaN para no romper el código)
df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')

# 3. Como el CLV original parece estar multiplicado por 100 (por el %), lo dividimos
df['customer_lifetime_value'] = df['customer_lifetime_value'] / 100

df['customer_lifetime_value'].head()

0           NaN
1     6979.5359
2    12887.4317
3     7645.8618
4     5363.0765
Name: customer_lifetime_value, dtype: float64

In [75]:
# 1. Extraemos el número que está entre las barras / / 
# Si el dato es '1/0/00', nos quedamos con el '0'
def clean_complaints(val):
    if isinstance(val, str):
        return val.split('/')[1]
    return val

df['number_of_open_complaints'] = df['number_of_open_complaints'].apply(clean_complaints)

# 2. Convertimos a numérico
df['number_of_open_complaints'] = pd.to_numeric(df['number_of_open_complaints'], errors='coerce')

print("Limpieza terminada. Ahora las quejas son números:")
print(df['number_of_open_complaints'].unique())

Limpieza terminada. Ahora las quejas son números:
[0 2 1 3 5 4]


In [76]:
# Verificamos el tamaño actual del DataFrame
shape = df.shape
print(f"El dataset tiene {shape[0]} filas y {shape[1]} columnas.")

El dataset tiene 1071 filas y 11 columnas.


In [77]:
# Revisamos los tipos de datos actuales
print(df.dtypes)

customer                      object
st                            object
gender                        object
education                     object
customer_lifetime_value      float64
income                       float64
monthly_premium_auto         float64
number_of_open_complaints      int64
policy_type                   object
vehicle_class                 object
total_claim_amount           float64
dtype: object


In [78]:
# Contamos valores únicos por columna
unique_counts = df.nunique()
print(unique_counts)

customer                     1071
st                              8
gender                          5
education                       6
customer_lifetime_value      1027
income                        774
monthly_premium_auto          132
number_of_open_complaints       6
policy_type                     3
vehicle_class                   6
total_claim_amount            761
dtype: int64


In [79]:
# Resumen estadístico de las columnas numéricas
df.describe().T

,count,mean,std,min,25%,50%,75%,max
customer_lifetime_value,1068.0,7936.902593,6434.784103,2004.350700,4034.079950,5881.742350,8962.871975,58166.553500
income,1071.0,39295.701214,30469.427060,0.000000,14072.000000,36234.000000,64631.000000,99960.000000
monthly_premium_auto,1071.0,193.234360,1601.190369,61.000000,68.000000,83.000000,109.500000,35354.000000
number_of_open_complaints,1071.0,0.394024,0.883769,0.000000,0.000000,0.000000,0.000000,5.000000
total_claim_amount,1071.0,404.986909,293.027260,0.382107,202.157702,354.729129,532.800000,2893.239678


In [80]:
# Resumen de las columnas tipo 'objeto' (texto)
df.describe(include='O').T

,count,unique,top,freq
customer,1071,1071,RB50392,1
st,1071,8,Oregon,320
gender,954,5,F,457
education,1071,6,Bachelor,324
policy_type,1071,3,Personal Auto,780
vehicle_class,1071,6,Four-Door Car,576


## Challenge 2: analyzing the data

### Exercise 1

The marketing team wants to know the top 5 less common customer locations. Create a pandas Series object that contains the customer locations and their frequencies, and then retrieve the top 5 less common locations in ascending order.

In [82]:
# 1. Creamos la serie con las frecuencias de cada estado
location_series = df['st'].value_counts()

# 2. Ordenamos de forma ascendente (menor a mayor) y tomamos los primeros 5
bottom_5 = location_series.sort_values(ascending=True).head(5)

print("Las 5 ubicaciones menos comunes son:")
print(bottom_5)

Las 5 ubicaciones menos comunes son:
st
AZ             25
WA             30
Washington     81
Nevada         98
Cali          120
Name: count, dtype: int64


### Exercise 2

The sales team wants to know the total number of policies sold for each type of policy. Create a pandas Series object that contains the policy types and their total number of policies sold, and then retrieve the policy type with the highest number of policies sold.

*Hint:*
- *Using value_counts() method simplifies this analysis.*
- *Futhermore, there is a method that returns the index of the maximum value in a column or row.*


In [83]:
# 1. Contamos las ventas por tipo de póliza
policy_counts = df['policy_type'].value_counts()

# 2. Identificamos el tipo de póliza con más ventas
top_policy = policy_counts.idxmax()

print(f"La póliza más vendida es: {top_policy} con {policy_counts.max()} ventas.")

La póliza más vendida es: Personal Auto con 780 ventas.


### Exercise 3

The sales team wants to know if customers with Personal Auto have a lower income than those with Corporate Auto. How does the average income compare between the two policy types?

- Use *loc* to create two dataframes: one containing only Personal Auto policies and one containing only Corporate Auto policies.
- Calculate the average income for each policy.
- Print the results.

In [84]:
# 1. Creamos DataFrames separados
personal_df = df.loc[df['policy_type'] == 'Personal Auto']
corporate_df = df.loc[df['policy_type'] == 'Corporate Auto']

# 2. Calculamos promedios
mean_personal = personal_df['income'].mean()
mean_corporate = corporate_df['income'].mean()

print(f"Ingreso promedio Personal Auto: {mean_personal:.2f}")
print(f"Ingreso promedio Corporate Auto: {mean_corporate:.2f}")

Ingreso promedio Personal Auto: 38180.70
Ingreso promedio Corporate Auto: 41390.31


### Bonus: Exercise 4


Your goal is to identify customers with a high policy claim amount.

Instructions:

- Review again the statistics for total claim amount to gain an understanding of the data.
- To identify potential areas for improving customer retention and profitability, we want to focus on customers with a high policy claim amount. Consider customers with a high policy claim amount to be those in the top 25% of the total claim amount. Create a pandas DataFrame object that contains information about customers with a policy claim amount greater than the 75th percentile.
- Use DataFrame methods to calculate summary statistics about the high policy claim amount data. 

*Note: When analyzing data, we often want to focus on certain groups of values to gain insights. Percentiles are a useful tool to help us define these groups. A percentile is a measure that tells us what percentage of values in a dataset are below a certain value. For example, the 75th percentile represents the value below which 75% of the data falls. Similarly, the 25th percentile represents the value below which 25% of the data falls. When we talk about the top 25%, we are referring to the values that fall above the 75th percentile, which represent the top quarter of the data. On the other hand, when we talk about the bottom 25%, we are referring to the values that fall below the 25th percentile, which represent the bottom quarter of the data. By focusing on these groups, we can identify patterns and trends that may be useful for making decisions and taking action.*

*Hint: look for a method that gives you the percentile or quantile 0.75 and 0.25 for a Pandas Series.*

*Hint 2: check `Boolean selection according to the values of a single column` in https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9*

In [85]:
# 1. Calculamos el valor de corte (Percentil 75)
p75 = df['total_claim_amount'].quantile(0.75)

# 2. Filtramos los clientes que superan ese umbral
high_claim_df = df[df['total_claim_amount'] > p75]

# 3. Resumen estadístico de este grupo específico
print(f"Valor mínimo para ser considerado 'Reclamo Alto': {p75}")
print(high_claim_df['total_claim_amount'].describe())

Valor mínimo para ser considerado 'Reclamo Alto': 532.8
count     264.000000
mean      782.228263
std       292.751640
min       537.600000
25%       606.521741
50%       679.597985
75%       851.400000
max      2893.239678
Name: total_claim_amount, dtype: float64
